In [22]:
# Cell 1: 安装依赖（如果还没装）
# pip install openai

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from openai import OpenAI
import os

In [23]:
# Cell 2: 准备文档和向量库（和昨天一样）
with open("test_rag.txt", "w") as f:
    f.write("PyTorch是一个深度学习框架。\nONNX用于模型转换。\nTensorRT用于推理加速。\n")

loader = TextLoader("test_rag.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=20, chunk_overlap=0, separator="\n", keep_separator=False)
docs = text_splitter.split_documents(documents)
print([doc.page_content for doc in docs])

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore.delete_collection()
vectorstore = Chroma.from_documents(docs, embeddings)
print(vectorstore._collection.count())

['PyTorch是一个深度学习框架。', 'ONNX用于模型转换。', 'TensorRT用于推理加速。']
3


In [24]:
# Cell 3: 配置 DeepSeek API（免费）
api_key = os.environ.get('DEEPSEEK_API_KEY') or os.environ.get('OPENAI_API_KEY')
# print(api_key)
if not api_key:
    raise ValueError("No API key found. Set DEEPSEEK_API_KEY or OPENAI_API_KEY")

client = OpenAI(
    api_key=api_key,  # 去 platform.deepseek.com 注册获取
    base_url="https://api.deepseek.com"
)

In [26]:
# Cell 4: RAG 问答函数
def rag_answer(query):
    # 1. 检索
    results = vectorstore.similarity_search(query, k=1)
    context = results[0].page_content
    
    # 2. 构造 prompt
    prompt = f"""只根据以下知识库内容回答问题，不要使用网上的知识。如果知识库中没有相关信息，请说"不知道"。

知识库：{context}

问题：{query}

回答："""
    
    # 3. 调用 LLM
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    return response.choices[0].message.content

In [27]:
# Cell 5: 测试
query = "什么是TensorRT？"
answer = rag_answer(query)
print(f"问题：{query}")
print(f"回答：{answer}")

问题：什么是TensorRT？
回答：TensorRT是用于推理加速的工具。


## 用不同文档测试

In [38]:
# 换一个文档
with open("langchain_intro.txt", "w") as f:
    f.write("LangChain是一个用于开发大语言模型应用的框架。\n")
    f.write("它提供了Chain、Agent、Memory等模块。\n")
    f.write("RAG是LangChain最常用的应用场景之一。\n")

# 重复上面的分块、向量化、检索、LLM生成流程
loader = TextLoader("langchain_intro.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=30, chunk_overlap=0, separator="\n", keep_separator=False)
docs = text_splitter.split_documents(documents)
print([doc.page_content for doc in docs])

vectorstore_new.delete_collection()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore_new = Chroma.from_documents(docs, embeddings)
print(vectorstore_new._collection.count())

['LangChain是一个用于开发大语言模型应用的框架。', '它提供了Chain、Agent、Memory等模块。', 'RAG是LangChain最常用的应用场景之一。']
3


In [53]:
# Cell 4: RAG 问答函数
def rag_answer_new(query):
    # 1. 检索
    results = vectorstore_new.similarity_search_with_score(query, k=2)
    for doc, score in results:
        print(f"分数: {score:.4f} | 内容: {doc.page_content}")
    docs = [doc for doc, score in results]  # 或者 [result[0] for result in results]
    context = "\n".join([doc.page_content for doc in docs])

    # 2. 打印调试信息（可选）
    print(f"检索到的chunk:")
    for i, r in enumerate(docs):
        print(f"  {i+1}: {r.page_content}")
    
    # 2. 构造 prompt
    prompt = f"""只根据以下知识库内容回答问题，不要使用网上的知识。如果知识库中没有相关信息，请说"不知道"。

知识库：{context}

问题：{query}

回答："""
    
    # 3. 调用 LLM
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    return response.choices[0].message.content

In [55]:
# Cell 5: 测试
query = "LangChain提供了什么模块？例如memory？"
answer = rag_answer_new(query)
print(f"问题：{query}")
print(f"回答：{answer}")

分数: 0.6035 | 内容: 它提供了Chain、Agent、Memory等模块。
分数: 0.6726 | 内容: LangChain是一个用于开发大语言模型应用的框架。
检索到的chunk:
  1: 它提供了Chain、Agent、Memory等模块。
  2: LangChain是一个用于开发大语言模型应用的框架。
问题：LangChain提供了什么模块？例如memory？
回答：根据知识库内容，LangChain提供了Chain、Agent、Memory等模块。
